In [ ]:
from option_analyzer import *
self = OptionAnalyzer('quotes', 'chain')

### Either run these two cells

### Or run this cell to read from data directory

In [ ]:
option_type = 'put'
servers = sorted(set([f.split('~')[1] for f in glob(os.path.expanduser(f'~/lab/data/{option_type}~*~*.csv'))]))
latest_option_files = [sorted(glob(os.path.expanduser(f'~/lab/data/{option_type}~{svr}~*.csv')))[-1] for svr in servers]
print('\n'.join(map(os.path.basename, latest_option_files)))
chain_file_mtimes = dict([(os.path.basename(_f), os.path.getmtime(_f)) for _f in glob(os.path.expanduser('~/lab/chain/*'))])
latest_symbol = sorted(chain_file_mtimes, key=chain_file_mtimes.get)[-1]
print('Last symbol:', latest_symbol, datetime.fromtimestamp(chain_file_mtimes[latest_symbol]).strftime('%F %T'))
dfp = pd.concat([pd.read_csv(_f) for _f in latest_option_files])

### Put options with no earning date on or before expiration date
Sell puts to maximize hdteProfit

In [ ]:
hdte_resid_ub = 0.5
spread_ub = 25
moneyness_ub = 0.9
premium_lb = 0.5
delta_lb = -0.3
_filter = (dfp.moneyness <= moneyness_ub) & (dfp.pctSpread <= spread_ub) & (dfp.hdte_resid<=hdte_resid_ub)
_filter = _filter & (dfp.mid >= premium_lb) & (dfp.Delta >= delta_lb)
_filter = _filter & (~dfp.symbol.str.contains('HIMS|SNDK'))
#_filter = _filter & (dfp.E.isna() |(dfp.E > dfp.dte)) # Note: Fidelity's earning report dates are not reliable
_dfp = dfp[_filter].sort_values(by='hdteProfit', ascending=False)
#_filter = _filter & (dfp.hdteProfit >= 20) & (dfp.strike <= 150)
#_dfp = dfp[_filter].sort_values(by='dth')
print('Options after the filters:', _dfp.shape[0])
px.scatter(_dfp.head(120), x='moneyness', y='hdteProfit', color='symbol', height=500).show()
_dfp.head(60)

### Put options for specific symbols

In [ ]:
_filter = dfp.symbol.str.contains('QQQ') & (dfp.moneyness >= 0.9) & (dfp.moneyness <= 1) & (dfp.dte <= 60) & (dfp.mid >= 1.3)
#_filter = _filter & (dfp.expDt == '2026-03-31')
_dfp = dfp[_filter].sort_values(by='hdteProfit', ascending=False).head(200)
px.scatter(_dfp, x='strike', y='mid', color='expDt', height=500).show()
px.scatter(_dfp[_dfp.expDt=='2026-03-31'], x='strike', y='mid', color='hdteProfit', height=500).show()
print(_dfp.shape)
_dfp.head(25)

### Put options: top 500 in-the-money

In [ ]:
px.scatter(dfp[(dfp.moneyness <= 1) & (dfp.hdteProfit <= 100)].sort_values(by='hdteProfit', ascending=False).head(20), x='hdte_resid', y='hdteProfit', color='symbol', height=600)